### local descriptor extraction using DiNOv2 w/ registers

export PYTHONPATH=${PWD}:$PYTHONPATH

python extract/extract_descriptors.py --dataset ilias \
                              --backbone dinov2 \
                              --save_path data \
                              --data_path /private/chechik-lab/ofirk/ILIAS/ilias_core \
                              --split _gallery \
                              --file_name /cortex/users/edenzon/codes/ames/data/ilias/test_gallery.txt \
                              --desc_type "local" \
                              --pretrained

python extract/extract_descriptors.py --dataset ilias \
                              --backbone dinov2 \
                              --save_path data \
                              --data_path /private/chechik-lab/ofirk/ILIAS/ilias_core \
                              --split _query \
                              --file_name /cortex/users/edenzon/codes/ames/data/ilias/test_query.txt \
                              --desc_type "local" \
                              --pretrained

### creating the nn index

python extract/prepare_topk_global.py \
  --dataset ilias \
  --desc_name dinov3 \
  --data_root ${PWD}/data


### run ames inference

python src/run_ames_inference.py \
  --dataset ilias \
  --global_desc dinov3 \
  --local_desc dinov2 \
  --topk 100

### evaluate
python3 -u src/evaluate.py --multirun \
        descriptors=dinov2 \
        data_root=data \
        model_path=dinov2_ames.pt \
        model.binarized=False \
        dataset@test_dataset=roxford \
        test_dataset.query_desc_num=600 \
        test_dataset.db_desc_num=600 \
        test_dataset.batch_size=300 \
        test_dataset.lamb=[0.55] \
        test_dataset.temp=[0.3] \
        test_dataset.num_rerank=[1600]

In [2]:
from pathlib import Path
import shutil

src = Path("/cortex/users/edenzon/codes/ames/data/ilias/dinov2_gallery_local.hdf5")
dst_dir = Path("/private/chechik-lab/ofirk/ILIAS/features/AMES")
dst_dir.mkdir(parents=True, exist_ok=True)
dst = dst_dir / src.name

shutil.copy2(src, dst)
print(f"Copied {src} -> {dst}")

Copied /cortex/users/edenzon/codes/ames/data/ilias/dinov2_gallery_local.hdf5 -> /private/chechik-lab/ofirk/ILIAS/features/AMES/dinov2_gallery_local.hdf5
